In [8]:
import scanpy as sc
import tangram as tg

In [2]:
ad_sc = sc.read_h5ad('../../data/linnarsson_adolescence_full.h5ad')
ad_sp = sc.read_h5ad('../../data/ST_BRICHOS_region_subcluster.h5ad')

/Users/christoffer/miniconda3/envs/sc/lib/python3.8/site-packages/anndata/_core/anndata.py:1838: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/Users/christoffer/miniconda3/envs/sc/lib/python3.8/site-packages/anndata/_core/anndata.py:1840: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [6]:
ad_sp.X = ad_sp.layers['counts']

In [9]:
tg.pp_adatas(ad_sc, ad_sp)

INFO:root:16562 training genes are saved in `uns``training_genes` of both single cell and spatial Anndatas.
INFO:root:16563 overlapped genes are saved in `uns``overlap_genes` of both single cell and spatial Anndatas.
INFO:root:uniform based density prior is calculated and saved in `obs``uniform_density` of the spatial Anndata.
INFO:root:rna count based density prior is calculated and saved in `obs``rna_count_based_density` of the spatial Anndata.


In [13]:
import torch
import tangram as tg

# Detect MPS device
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

# Then call Tangram normally
tg.map_cells_to_space(
    ad_sc,
    ad_sp,
    device=device,   # this enables MPS
    mode='clusters', # or 'cells'
    cluster_label = 'Class',
    num_epochs=250,
)

Using device: mps


INFO:root:Allocate tensors for mapping.
INFO:root:Begin training with 16562 genes and rna_count_based density_prior in clusters mode...
INFO:root:Printing scores every 100 epochs.


Score: 0.308, KL reg: 0.273
Score: 0.473, KL reg: 0.000
Score: 0.473, KL reg: 0.000


INFO:root:Saving results..


AnnData object with n_obs × n_vars = 7 × 25660
    obs: 'Class', 'cluster_density'
    var: 'in_tissue', 'array_row', 'array_col', 'pxl_row_in_fullres', 'pxl_col_in_fullres', 'sample', 'sample_id', 'n_genes', 'leiden', 'treatment', 'barcode', 'region_annotation', 'leiden_0.5', 'leiden_0.75', 'leiden_1', 'leiden_1.5', 'leiden_2', 'leiden_2.5', 'PIG_score', 'OLIG_score', 'leiden_0.1', 'leiden_0.2', 'leiden_0.3', 're_annotation_regions', 'uniform_density', 'rna_count_based_density'
    uns: 'train_genes_df', 'training_history'

In [14]:
tg.project_cell_annotations(ad_map, ad_sp, annotation='Class')
annotation_list = list(pd.unique(ad_sc.obs['Class']))

NameError: name 'ad_map' is not defined

In [ ]:
df = ad_sp.obsm["tangram_ct_pred"]              # spots × classes (probabilities)


In [ ]:
row_sums = df.sum(axis=1)
df_norm = df.div(row_sums.replace(0, np.nan), axis=0).fillna(0.0)

# now rows sum to 1
assert np.allclose(df_norm.sum(axis=1).to_numpy(), 1.0, atol=1e-6)

In [ ]:
calls = df_norm.idxmax(axis=1)

In [ ]:
calls.value_counts()

In [ ]:
ad_sp.obs['call'] = ad_sp.obs.index.map(calls)

In [ ]:
sc.pl.umap(ad_sp, color = 'call')

In [ ]:
calls = df_grp.idxmax(axis=1)
pmax  = df_grp.max(axis=1)
ad_sp.obs["tg_call"]  = pd.Categorical(calls)
ad_sp.obs["tg_pmax"]  = pmax

In [ ]:
df_norm

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "utils")))
from analysis_utils import (
    volcano_plot_br,
    plot_spatial_clusters_per_sample,
    plot_dotplot_by_treatment,
    plot_relative_cluster_composition,
    score_and_plot_modules
)
plot_spatial_clusters_per_sample(ad_sp, color = 'call', figsize=(20,15))

In [ ]:
import numpy as np
import scanpy as sc

def plot_cell_annotation_sc_with_library(
    adata_sp,
    annotation_list,
    *,
    library_id=None,        # e.g. "P24215_101"
    library_col=None,       # e.g. "sample_id" if each cell has its library label
    x="x",
    y="y",
    spot_size=None,
    scale_factor=None,
    perc=0,
    alpha_img=1.0,
    bw=False,
    ax=None,
):
    """
    Drop-in replacement for tg.plot_cell_annotation_sc with support for multi-library AnnData.
    """

    # Clean previous columns
    adata_sp.obs.drop(annotation_list, inplace=True, errors="ignore", axis=1)

    # Build obs columns from obsm["tangram_ct_pred"]
    if "tangram_ct_pred" not in adata_sp.obsm:
        raise KeyError("adata_sp.obsm['tangram_ct_pred'] not found.")
    df = adata_sp.obsm["tangram_ct_pred"][annotation_list]
    # --- construct_obs_plot(df, adata_sp, perc=perc) inline ---
    # keep top-`perc` percentile if desired (perc=0 keeps all)
    if perc and perc > 0:
        thr = np.nanpercentile(df.values, 100 - perc, axis=0)
        df = df.where(df.ge(thr, axis=1), other=np.nan)
    for col in df.columns:
        adata_sp.obs[col] = df[col].values

    # Decide plotting path
    has_uns_spatial = "spatial" in adata_sp.uns and isinstance(adata_sp.uns["spatial"], dict)

    # Case A: Visium-style with .uns['spatial'] present -> must specify library
    if has_uns_spatial:
        if library_id is None:
            # Try to infer from a per-cell column if provided
            if library_col is not None and library_col in adata_sp.obs:
                vals = adata_sp.obs[library_col].astype(str).unique().tolist()
                if len(vals) == 1:
                    library_id = vals[0]
                else:
                    raise ValueError(
                        "Found multiple libraries in `.uns['spatial']` and multiple values in "
                        f"obs['{library_col}'] ({vals}). Please pass library_id='...'. "
                        f"Options in uns['spatial']: {list(adata_sp.uns['spatial'].keys())}"
                    )
            else:
                raise ValueError(
                    "Found multiple libraries in `.uns['spatial']`. Please pass library_id='...'. "
                    f"Options: {list(adata_sp.uns['spatial'].keys())}"
                )

        # Rules from original Tangram function
        if spot_size is not None and scale_factor is not None:
            raise ValueError("Spot Size and Scale Factor should be None when ad_sp.uns['spatial'] exists")

        sc.pl.spatial(
            adata_sp,
            color=annotation_list,
            library_id=library_id,    # <-- the missing piece
            cmap="viridis",
            show=False,
            frameon=False,
            spot_size=spot_size,
            scale_factor=scale_factor,
            alpha_img=alpha_img,
            bw=bw,
            ax=ax,
        )

    # Case B: No Visium mapping in .uns -> fall back to coordinates
    else:
        # If there are no spatial coords in .obsm, build them from obs[x], obs[y]
        if "spatial" not in adata_sp.obsm:
            if (x not in adata_sp.obs) or (y not in adata_sp.obs):
                raise KeyError(
                    "No .uns['spatial'] and .obsm['spatial'] missing. "
                    f"Provide obs columns x='{x}' and y='{y}' or supply a Visium .uns['spatial']."
                )
            coords = np.column_stack([adata_sp.obs[x].to_numpy(), adata_sp.obs[y].to_numpy()])
            adata_sp.obsm["spatial"] = coords.astype(float)

        if spot_size is None and scale_factor is None:
            raise ValueError(
                "Spot Size and Scale Factor cannot be None when ad_sp.uns['spatial'] does not exist"
            )

        sc.pl.spatial(
            adata_sp,
            color=annotation_list,
            cmap="viridis",
            show=False,
            frameon=False,
            spot_size=spot_size,
            scale_factor=scale_factor,
            alpha_img=alpha_img,
            bw=bw,
            ax=ax,
        )

    # cleanup
    adata_sp.obs.drop(annotation_list, inplace=True, errors="ignore", axis=1)